# 004 · Batch Inference & Evaluation

**How to run:**
1. Set `CACHE_FILES` in Section 2 — one entry per model run.
2. `Runtime → Run all`.

Evaluation features:
- Per-doc-type charts (heatmap, Exact Match, CER)
- Per-field metrics down to smallest leaf field
- Visual diff with colour-coded cells (green = match, red = mismatch)

In [ ]:

CHECKPOINT_OVERRIDE: str | None = None
# CHECKPOINT_OVERRIDE = "models/finetune/v11-20260529-065340/checkpoint-60"

print("Ready")


## 2 · Configuration

In [ ]:
import os, sys, json, re
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["USE_HF"] = "1"

BASE_DIR    = Path(os.path.abspath("."))
DATASET_DIR = BASE_DIR / "data" / "swift_dataset" 
IMAGES_DIR  = DATASET_DIR / "images"

# ── Load shared config ──────────────────────────────────────────────────────
sys.path.insert(0, str(BASE_DIR))
from pipeline_config import (
    EXCLUDED_FIELDS, DATE_NORMALIZE_EXCLUDE,
    NULL_PCT_THRESHOLD, LENGTH_MEAN_THRESHOLD,
    MAX_CARDHOLDERS, CARDHOLDER_FIELDS,
)

# Cache files: one entry per model run 
# Format: (cache_file_path, "human label")
# Use "auto" to derive label from checkpoint path.
CACHE_FILES = [
    (BASE_DIR / "rows_cache.json", "auto"),
    # (BASE_DIR / "rows_cache_v2.json", "v2-experiment"),
]

# Auto-detect latest checkpoin
def _is_qwen_vl_checkpoint(p: Path) -> bool:
    cfg = p / "adapter_config.json"
    if not cfg.exists():
        return False
    base = json.load(open(cfg)).get("base_model_name_or_path", "")
    return "Qwen2-VL" in base or "Qwen2.5-VL" in base


def _ckpt_sort_key(p: Path) -> tuple[int, int]:
    v = re.search(r"v(\d+)-", str(p))
    s = re.search(r"checkpoint-(\d+)$", str(p))
    return (int(v.group(1)) if v else 0, int(s.group(1)) if s else 0)


if CHECKPOINT_OVERRIDE:
    CHECKPOINT  = str(BASE_DIR / CHECKPOINT_OVERRIDE) if not Path(CHECKPOINT_OVERRIDE).is_absolute() else CHECKPOINT_OVERRIDE
    MODEL_NAME  = Path(CHECKPOINT).parent.name
    print(f"Using override checkpoint: {CHECKPOINT}")
else:
    ckpts = sorted(
        [p for p in (BASE_DIR / "models" / "finetune").glob("*/checkpoint-*")
         if _is_qwen_vl_checkpoint(p)],
        key=_ckpt_sort_key,
    )
    if not ckpts:
        raise FileNotFoundError("No valid checkpoint found in models/finetune/")
    CHECKPOINT = str(ckpts[-1])
    MODEL_NAME = Path(CHECKPOINT).parent.name
    print(f"Auto-selected latest checkpoint: {CHECKPOINT}")

print(f"MODEL_NAME : {MODEL_NAME}")
print("\nAll valid checkpoints:")
for ck in ckpts if not CHECKPOINT_OVERRIDE else [Path(CHECKPOINT)]:
    marker = "  ← ACTIVE" if str(ck) == CHECKPOINT else ""
    print(f"  {ck.parent.name}/{ck.name}{marker}")


print(f"BASE_DIR    : {BASE_DIR}")
print(f"CHECKPOINT  : {CHECKPOINT}")
print(f"MODEL_NAME  : {MODEL_NAME}")
print(f"CACHE_FILES : {[(str(c), l) for c, l in CACHE_FILES]}")


## 3 · Clear VRAM

In [ ]:
import gc, torch

for var in ["model", "trainer", "engine", "optimizer"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

free  = torch.cuda.mem_get_info()[0] / 1024**3
total = torch.cuda.mem_get_info()[1] / 1024**3
print(f"VRAM free: {free:.1f} GB / {total:.1f} GB")
if free < 8:
    print("WARNING: < 8 GB free — inference may OOM")


## 4 · Helper Functions

In [ ]:
import re, json
import numpy as np
import pandas as pd
from datetime import datetime
from dateutil import parser as dtparser
from dateutil.parser import ParserError

#String helpers 

def safe_str(value) -> str:
    """Convert any value to string; None → empty string."""
    if value is None:
        return ""
    if isinstance(value, list):
        return json.dumps(value, ensure_ascii=False)
    return str(value).strip()


# Flatten 

def flatten_cardholders(cardholders: list, sep: str = "_") -> dict:
    """Expand cardholder list into flat fields: cardholders_1_first_name, etc."""
    result = {}
    for i, holder in enumerate(cardholders[:MAX_CARDHOLDERS], start=1):
        if not isinstance(holder, dict):
            continue
        for field in CARDHOLDER_FIELDS:
            result[f"cardholders{sep}{i}{sep}{field}"] = holder.get(field)
    return result


def flatten_dict(d: dict, parent_key: str = "", sep: str = "_") -> dict:
    """
    Recursively flatten a nested dict.
    Lists of dicts (cardholders) → expand via flatten_cardholders.
    Plain lists → JSON string.
    """
    result = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            result.update(flatten_dict(v, new_key, sep))
        elif isinstance(v, list) and v and isinstance(v[0], dict):
            if "cardholder" in new_key.lower():
                result.update(flatten_cardholders(v, sep))
            else:
                result[new_key] = json.dumps(v, ensure_ascii=False)
        else:
            result[new_key] = v
    return result


# Date normalisation

DATE_KEYWORDS = {"date","dob","expiry","expire","issued","issue","birth","valid","from","start","end"}
BIRTH_EXCLUDE = {"place"}

def is_date_field(name: str) -> bool:
    """Return True if field name suggests a date value."""
    tokens = set(re.split(r"[_.]", name.lower()))
    if "birth" in tokens and tokens & BIRTH_EXCLUDE:
        tokens.discard("birth")
    return bool(tokens & DATE_KEYWORDS)


def try_parse_date(value: str) -> str:
    """Normalise a date string to YYYY-MM-DD; return original on failure."""
    if not value or not isinstance(value, str):
        return value
    v = value.strip()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", v):
        return v
    for pattern, fmt in [
        (r"^(\d{1,2})/(\d{1,2})/(\d{4})$", "%d/%m/%Y"),
        (r"^(\d{4})/(\d{2})/(\d{2})$",      "%Y/%m/%d"),
        (r"^(\d{1,2})-(\d{1,2})-(\d{4})$",  "%d-%m-%Y"),
    ]:
        if re.fullmatch(pattern, v):
            try:
                return datetime.strptime(v, fmt).strftime("%Y-%m-%d")
            except ValueError:
                pass
    try:
        return dtparser.parse(v, dayfirst=True).strftime("%Y-%m-%d")
    except Exception:
        return v


CATEGORY_MAP = {
    "victoria australia": "VIC",
    "victoria":           "VIC",
    "driver licence":     "AUS_DRIVER_LICENSE",
}
NUMERIC_KW = {"total", "amount", "kwh", "usage", "gst"}


def normalize_field_value(key: str, value: str) -> str:
    """Normalise a single field value (category map, numeric strip, uppercase)."""
    if not isinstance(value, str) or not value:
        return str(value) if value is not None else ""
    v = value.strip()
    if v.lower() in CATEGORY_MAP:
        return CATEGORY_MAP[v.lower()]
    if any(kw in key.lower() for kw in NUMERIC_KW):
        return re.sub(r"[a-zA-Z\s\$,]+$", "", v).strip()
    return v.upper()


def normalize_record(record: dict) -> dict:
    """
    Normalise a flat record dict in-place:
    - date fields → ISO 8601 (skip DATE_NORMALIZE_EXCLUDE)
    - other strings → normalize_field_value
    """
    for k, v in record.items():
        if not isinstance(v, str):
            continue
        if k in DATE_NORMALIZE_EXCLUDE:
            continue
        if is_date_field(k):
            record[k] = try_parse_date(v)
        else:
            record[k] = normalize_field_value(k, v)
    return record


print("Helper functions ready.")


## 5 · Load Cache or Run Inference

In [ ]:
from utils.evaluation import find_and_parse_json

def _is_valid_pred(pred: dict) -> bool:
    """Check prediction is non-empty and contains real values."""
    return (
        isinstance(pred, dict)
        and len(pred) > 0
        and any(v is not None and v not in ("...", " ") for v in pred.values())
    )

def _run_live_inference(checkpoint_path: str) -> list[dict]:
    """Run swift infer_main on the test set and return parsed rows."""
    from swift import infer_main

    test_file   = DATASET_DIR / "conversations_test_swift_format.json"
    result_file = BASE_DIR / "infer_results.jsonl"
    if result_file.exists():
        result_file.unlink()

    infer_main([
        "--adapters",       checkpoint_path,
        "--val_dataset",    str(test_file),
        "--result_path",    str(result_file),
        "--max_length",     "8192",
        "--max_pixels",     "8262144",
        "--max_new_tokens", "1024",
        "--temperature",    "0",
        "--infer_backend",  "transformers",
        "--load_data_args", "false",
        "--use_hf",         "true",
    ])

    test_samples = json.load(open(test_file))
    raw_lines    = [
        json.loads(l) for l in result_file.read_text().splitlines() if l.strip()
    ]
    model_label = Path(checkpoint_path).parent.name

    rows = []
    for idx, sample in enumerate(test_samples):
        # Mã thêm vào để in prompt và ground truth của sample đầu tiên nhằm kiểm tra label null
        if idx == 0:
            print("\n=== DEBUG KIỂM TRA PROMPT VÀ JSON LABEL NULL ===")
            print("System Message:", sample["messages"][0]["content"])
            print("User Prompt:", sample["messages"][1]["content"])
            print("Ground Truth (JSON):", sample["messages"][2]["content"])
            print("================================================\n")

        gt      = json.loads(sample["messages"][2]["content"])
        raw_out = raw_lines[idx]["response"] if idx < len(raw_lines) else ""
        pred    = find_and_parse_json(raw_out)
        valid   = _is_valid_pred(pred)
        rows.append({
            "image"      : sample["images"][0],
            "doc_type"   : gt.get("document_type"),
            "gt"         : gt,
            "pred"       : {k: v for k, v in pred.items() if not k.startswith("_")} if valid else {},
            "pred_raw"   : raw_out if not valid else "",
            "valid_json" : valid,
            "model"      : checkpoint_path,
            "model_label": model_label,
        })
    return rows

def load_or_run_inference(cache_path: Path, label: str) -> list[dict]:
    """Load rows from cache if it exists, otherwise run inference and save cache."""
    if cache_path.exists():
        print(f"Load from cache: {cache_path}")
        rows = json.load(open(cache_path))
        for row in rows:
            if label != "auto":
                row["model_label"] = label
            elif "model" in row:
                row["model_label"] = Path(row["model"]).parent.name
        
        # Thêm mã để in cấu trúc dữ liệu nếu đang load từ cache
        if rows:
            print("\n=== DEBUG KIỂM TRA DỮ LIỆU TỪ CACHE ===")
            print("Ground Truth Cache (Row 0):", json.dumps(rows[0].get("gt", {}), indent=2, ensure_ascii=False))
            print("=======================================\n")
            
        return rows

    print(f"No cache — running inference for '{label}'...")
    rows = _run_live_inference(CHECKPOINT)
    json.dump(rows, open(cache_path, "w"), ensure_ascii=False, indent=2)
    print(f"Cache saved: {cache_path}")
    return rows

# ── Load all model runs ──────────────────────────────────────────────────────
all_rows: list[dict] = []

for cache_path, label in CACHE_FILES:
    rows = load_or_run_inference(Path(cache_path), label)

    # Flatten gt/pred and normalise values
    for row in rows:
        row["gt_flat"]   = normalize_record(flatten_dict(row.get("gt", {})))
        row["pred_flat"] = normalize_record(flatten_dict(row.get("pred", {}))) if row["valid_json"] else {}

    valid = sum(r["valid_json"] for r in rows)
    lbl   = rows[0]["model_label"] if rows else label
    print(f"[{lbl}] {len(rows)} rows | Valid JSON: {valid}/{len(rows)}")
    all_rows.extend(rows)

print(f"\nTotal: {len(all_rows)} rows")

## 6 · Build Evaluation DataFrame

In [ ]:
import Levenshtein

# ── Edit distance encoding ───────────────────────────────────────────────────
#  -1 = missing ground truth   (GT empty)
#  -2 = predicted 'None'       (pred empty/None)
#  -3 = key missing             (field not in pred)
#  -4 = invalid JSON

DIST_LABEL_MAP = {
    -1: "missing groundtruth",
     0: "exact match",
     1: "1", 2: "2", 3: "3", 4: "4", 5: "5", 6: "6+",
     7: "predicted 'None'",
     8: "key missing",
     9: "invalid JSON",
}
HEATMAP_COLORS = [
    "#bababa","#66c2a5","#abdda4","#e6f598","#ffffbf",
    "#fee08b","#fdae61","#f46d43","#abd9e9","#74add1","#bebada",
]


def encode_dist(d: int) -> int:
    """Encode raw edit distance to heatmap bucket."""
    if d == -1:  return -1
    if d == -2:  return 7
    if d == -3:  return 8
    if d == -4:  return 9
    return min(d, 6)


def field_dist(gt_val: str, pred_val: str, valid: bool, in_pred: bool) -> int:
    """Compute encoded distance for a single (gt, pred) field pair."""
    if not valid:          return -4
    if not in_pred:        return -3
    if gt_val == "":       return -1
    if pred_val in ("", "None"): return -2
    return Levenshtein.distance(pred_val, gt_val)


def build_eval_df(rows: list[dict], excluded: set) -> pd.DataFrame:
    """Build per-(row, field) evaluation DataFrame from all_rows."""
    all_fields = sorted({k for r in rows for k in r["gt_flat"]} - excluded)

    records = []
    for row in rows:
        for field in all_fields:
            if field not in row["gt_flat"]:
                continue
            gt_val   = safe_str(row["gt_flat"].get(field))
            pred_val = safe_str(row["pred_flat"].get(field, "")) if row["valid_json"] else ""
            in_pred  = field in row["pred_flat"]
            dist     = field_dist(gt_val, pred_val, row["valid_json"], in_pred)
            records.append({
                "image"       : row.get("image", ""),
                "doc_type"    : row.get("doc_type", ""),
                "entity"      : field,
                "label_val"   : gt_val,
                "response_val": pred_val,
                "dist"        : dist,
                "valid_json"  : row["valid_json"],
                "model"       : row.get("model", CHECKPOINT),
                "pretty_name" : row.get("model_label", MODEL_NAME),
            })
    return pd.DataFrame(records)


df_multi = build_eval_df(all_rows, EXCLUDED_FIELDS)
print(f"df_multi: {df_multi.shape}")
print(f"Models  : {df_multi['pretty_name'].unique().tolist()}")
print(f"Doc types: {sorted(df_multi['doc_type'].dropna().unique().tolist())}")

# Fields with >90% empty GT
missing_gt = [
    f for f in df_multi["entity"].unique()
    if (df_multi[df_multi["entity"] == f]["label_val"] == "").mean() > 0.9
]
print(f"Missing GT fields ({len(missing_gt)}): {missing_gt}")


## 7 · Feature Categorisation

In [ ]:
from utils.entities import analyze_string_distribution_by_entity, categorize_features

first_model = df_multi["pretty_name"].unique()[0]
df_one      = df_multi[df_multi["pretty_name"] == first_model]

string_dist_df   = analyze_string_distribution_by_entity(df_one, col_value="label_val")
feature_cats_raw = categorize_features(
    string_dist_df,
    null_percentage_threshold=NULL_PCT_THRESHOLD,
    length_mean_threshold=LENGTH_MEAN_THRESHOLD,
)
# Normalise to string values
feature_categories: dict[str, str] = {
    e: (c.value if hasattr(c, "value") else c)
    for e, c in feature_cats_raw.items()
}

for cat in ["MISSING_GROUND_TRUTH", "SHORT_TEXT", "LONG_TEXT"]:
    members = [e for e, c in feature_categories.items() if c == cat]
    print(f"\n{cat} ({len(members)}): {members}")


## 8 · Edit Distance Heatmap — All Doc Types

In [ ]:
from plotnine import *
from IPython.display import Markdown, display

def _prepare_heatmap_df(df: pd.DataFrame) -> tuple[pd.DataFrame, list, list]:
    """Encode distances, sort by doc_type + entity_type, return df + order + boundaries."""
    df = df.copy()
    df["dist_encoded"] = df["dist"].apply(encode_dist)
    df["dist_label"]   = pd.Categorical(
        df["dist_encoded"].replace(DIST_LABEL_MAP),
        categories=list(DIST_LABEL_MAP.values()), ordered=True,
    )
    df["entity_type"]  = df["entity"].map(feature_categories)
    df["entity_label"] = df["doc_type"].fillna("?") + " | " + df["entity"]
    df = df.sort_values(["doc_type", "entity_type", "entity"])

    entity_order_df = (
        df[["entity_label", "entity_type", "doc_type"]]
        .drop_duplicates(subset=["entity_label"])
        .reset_index(drop=True)
    )
    ordered_labels = list(entity_order_df["entity_label"])
    boundaries = [
        i + 0.5
        for i in entity_order_df.index[
            (entity_order_df["entity_type"] != entity_order_df["entity_type"].shift())
            | (entity_order_df["doc_type"]    != entity_order_df["doc_type"].shift())
        ].tolist()[1:]
    ]
    df["pretty_name"] = pd.Categorical(
        df["pretty_name"], categories=sorted(df["pretty_name"].unique()), ordered=True
    )
    return df, ordered_labels, boundaries


df_heatmap, ordered_labels, boundaries = _prepare_heatmap_df(df_multi)
n_entities = len(ordered_labels)
height     = max(12, n_entities * 0.22)

display(Markdown("### All Entities"))
(
    ggplot(df_heatmap, aes(x="entity_label", fill="factor(dist_label)"))
    + geom_bar(position="stack", color="black", size=0.5)
    # + geom_bar(stat="identity", position="dodge", colour="gray")
    + facet_wrap("~pretty_name", scales="free")
    + labs(x="Doc Type | Field", y="Count", fill="Edit distance")
    + coord_flip()
    + scale_fill_manual(values=HEATMAP_COLORS, labels=list(DIST_LABEL_MAP.values()))
    + theme_minimal()
    + theme(figure_size=(16, height), legend_position="bottom", legend_direction="horizontal")
    + scale_x_discrete(limits=ordered_labels)
).show()



## 9 · Edit Distance Heatmap — Per Doc Type

In [ ]:
doc_types_present = sorted(df_heatmap["doc_type"].dropna().unique())

for dt in doc_types_present:
    df_dt = df_heatmap[df_heatmap["doc_type"] == dt]
    labels_dt = [l for l in ordered_labels if l.startswith(dt)]
    n = len(labels_dt)
    h = max(5, n * 0.3)

    display(Markdown(f"### {dt}"))
    (
        ggplot(df_dt, aes(x="entity_label", fill="factor(dist_label)"))
        + geom_bar(position="stack", color="black", size=0.4)
        + facet_wrap("~pretty_name", scales="free")
        + labs(x="Field", y="Count", fill="Edit distance")
        + coord_flip()
        + scale_fill_manual(values=HEATMAP_COLORS, labels=list(DIST_LABEL_MAP.values()))
        + theme_minimal()
        + theme(figure_size=(14, h), legend_position="bottom", legend_direction="horizontal")
        + scale_x_discrete(limits=labels_dt)
    ).show()


## 10 · Model Performance Metrics

In [ ]:
import pandas as pd
import evaluate

# Thêm "cer" vào danh sách load trực tiếp từ Hugging Face
_hf_metrics = {name: evaluate.load(name) for name in ["exact_match", "bleu", "rouge", "cer"]}

EMPTY_METRICS = {
    "exact_match": 0.0, "cer_score": 0.0,
    "bleu": 0.0, "rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0, "rougeLsum": 0.0,
}

def compute_metrics(refs: list[str], preds: list[str]) -> pd.Series:
    """
    Compute Exact Match, CER, BLEU, ROUGE for a list of (ref, pred) pairs.
    Skips pairs where ref is empty.
    """
    valid = [(str(r), str(p)) for r, p in zip(refs, preds) if str(r).strip()]
    if not valid:
        return pd.Series(EMPTY_METRICS)

    r_list, p_list = zip(*valid)
    result = {}

    for name, obj in _hf_metrics.items():
        try:
            # Tính toán metric
            res = obj.compute(predictions=list(p_list), references=list(r_list)) or {}
            
            # Thư viện trả về key "cer", ta map nó thành "cer_score" để giữ nguyên output gốc của bạn
            if name == "cer" and "cer" in res:
                result["cer_score"] = res["cer"]
            else:
                result.update(res)
                
        except Exception:
            # Gán giá trị 0.0 nếu có lỗi khi tính toán
            if name == "cer":
                result["cer_score"] = 0.0
            else:
                result[name] = 0.0

    return pd.Series(result)

print("Metrics ready:", list(_hf_metrics.keys()))

In [ ]:
relevant_entities = [e for e, c in feature_categories.items() if c != "MISSING_GROUND_TRUTH"]
df_relevant = df_multi[df_multi["entity"].isin(relevant_entities)].copy()

metrics_rows = []
for (model_path, model_name, entity), grp in df_relevant.groupby(["model", "pretty_name", "entity"]):
    res = compute_metrics(
        grp["label_val"].astype(str).tolist(),
        grp["response_val"].astype(str).tolist(),
    )
    row = res.to_dict()
    row.update({"model": model_path, "pretty_name": model_name, "entity": entity,
                "doc_type": grp["doc_type"].iloc[0] if len(grp) > 0 else ""})
    metrics_rows.append(row)

df_eval = pd.DataFrame(metrics_rows)
print(f"df_eval: {df_eval.shape}")
df_eval.head(3)


## 12 · Metrics Summary Table

In [ ]:
short_ents = [e for e, c in feature_categories.items() if c == "SHORT_TEXT"]
long_ents  = [e for e, c in feature_categories.items() if c == "LONG_TEXT"]

summary_rows = []
for (model_path, model_name), grp in df_eval.groupby(["model", "pretty_name"]):
    row = {"model": model_path, "pretty_name": model_name}
    short_grp = grp[grp["entity"].isin(short_ents)]
    if not short_grp.empty:
        row["exact_match"] = short_grp["exact_match"].mean()
        row["cer_score"]   = short_grp["cer_score"].mean()
    if long_ents:
        long_grp = grp[grp["entity"].isin(long_ents)]
        if not long_grp.empty:
            for col in ["rouge1", "rouge2", "rougeL"]:
                row[col] = long_grp[col].mean()
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
metric_cols = [c for c in ["exact_match", "cer_score", "rouge1", "rouge2", "rougeL"]
               if c in df_summary.columns]
df_summary[metric_cols] = df_summary[metric_cols].round(4)

# Highlight best values
def _hi_max(s): return ["background-color:#99d594" if v == s.max() else "" for v in s]
def _hi_min(s): return ["background-color:#99d594" if v == s.min() else "" for v in s]

higher = [c for c in metric_cols if c != "cer_score"]
lower  = ["cer_score"] if "cer_score" in metric_cols else []

styled = df_summary[["pretty_name"] + metric_cols].style
for col in higher:
    styled = styled.apply(_hi_max, subset=[col])
for col in lower:
    styled = styled.apply(_hi_min, subset=[col])

display(styled)


## 13 · Visual Diff — Colour-Coded

In [ ]:
import textwrap
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML

N_SAMPLES    = 20
COL_W        = 36

# Colours for HTML diff cells
CELL_OK   = "#d4edda"   # light green
CELL_MISS = "#f8d7da"   # light red
CELL_NULL = "#fff3cd"   # light yellow (pred missing/None)
CELL_HDR  = "#cfe2ff"   # header blue


def _cell(text: str, bg: str, width: str = "280px") -> str:
    """Build a coloured HTML table cell."""
    escaped = str(text).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    return f'<td style="background:{bg};padding:4px 8px;width:{width};font-size:12px;vertical-align:top;white-space:pre-wrap">{escaped}</td>'


def render_diff_html(gt_flat: dict, pred_flat: dict) -> str:
    """Render a GT vs Pred comparison table as HTML with colour-coded cells."""
    rows_html = [
        f'<tr>'
        f'{_cell("Field", CELL_HDR, "200px")}'
        f'{_cell("Ground Truth", CELL_HDR)}'
        f'{_cell("Prediction", CELL_HDR)}'
        f'</tr>'
    ]
    for field in sorted(gt_flat.keys()):
        gt_v   = safe_str(gt_flat.get(field))
        pred_v = safe_str(pred_flat.get(field, ""))
        match  = gt_v == pred_v

        if match:
            bg = CELL_OK
        elif pred_v in ("", "None"):
            bg = CELL_NULL
        else:
            bg = CELL_MISS

        icon = "✓" if match else "✗"
        rows_html.append(
            f'<tr>'
            f'{_cell(f"{icon} {field}", CELL_HDR, "200px")}'
            f'{_cell(gt_v, bg)}'
            f'{_cell(pred_v, bg)}'
            f'</tr>'
        )
    return f'<table style="border-collapse:collapse;font-family:monospace">{"".join(rows_html)}</table>'


def show_image(image_ref: str, idx: int, doc_type: str) -> None:
    """Display a document image inline."""
    p = Path(image_ref)
    if not p.is_absolute():
        p = IMAGES_DIR / p.name
    try:
        img = Image.open(p).convert("RGB")
        fig, ax = plt.subplots(figsize=(4, 6))
        ax.imshow(img); ax.axis("off")
        ax.set_title(f"#{idx+1} | {doc_type}", fontsize=10)
        plt.tight_layout(); plt.show()
    except Exception as e:
        print(f"  Image not found: {e}")


for idx, row in enumerate(all_rows[:N_SAMPLES]):
    gt_flat   = row["gt_flat"]
    pred_flat = row["pred_flat"] if row["valid_json"] else {}

    show_image(row["image"], idx, row.get("doc_type", "?"))

    status = "✓ valid JSON" if row["valid_json"] else "✗ invalid JSON"
    print(status)
    display(HTML(render_diff_html(gt_flat, pred_flat)))
    print()
